# predicting neurodevelopmental and MH conditions frorm c4

In [ ]:
# Imports for co-occurring conditions prediction
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("=== CO-OCCURRING CONDITIONS PREDICTION EXPERIMENT ===")

€ 2. load data and initial exploration 

In [ ]:
# Load the original C4 dataset with diagnosis information
df_raw = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/raw/data_c4_raw.csv')

print("=== DATASET EXPLORATION ===")
print(f"Original dataset shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")

# Remove test entries (first 16 rows)
df_raw = df_raw.iloc[16:].reset_index(drop=True)

# Remove records with missing compulsory data (opted out of data sharing)
compulsory_cols = ['age', 'sex', 'handedness', 'education', 'occupation', 'country_region']
df_clean = df_raw.dropna(subset=compulsory_cols)

print(f"After removing test entries and opt-outs: {df_clean.shape}")

# Explore diagnosis columns
diagnosis_cols = [col for col in df_clean.columns if 'diagnosis' in col]
print(f"Diagnosis columns: {diagnosis_cols}")

# Show diagnosis distribution
for col in diagnosis_cols:
    if col in df_clean.columns:
        print(f"{col} distribution:")
        print(df_clean[col].value_counts().head(10))

# 3. create autism only dataset

In [ ]:
# Create autism-only dataset
print("=== CREATING AUTISM-ONLY DATASET ===")

# Filter to autistic population only
autism_mask = (df_clean['diagnosis_0'] == 2) | (df_clean['diagnosis_1'] == 2) | (df_clean['diagnosis_2'] == 2) | \
              (df_clean['diagnosis_3'] == 2) | (df_clean['diagnosis_4'] == 2) | (df_clean['diagnosis_5'] == 2) | \
              (df_clean['diagnosis_6'] == 2) | (df_clean['diagnosis_7'] == 2) | (df_clean['diagnosis_8'] == 2) | \
              (df_clean['autism_diagnosis_0'] == 1) | (df_clean['autism_diagnosis_0'] == 2) | (df_clean['autism_diagnosis_0'] == 3) | \
              (df_clean['autism_diagnosis_1'] == 1) | (df_clean['autism_diagnosis_1'] == 2) | (df_clean['autism_diagnosis_1'] == 3) | \
              (df_clean['autism_diagnosis_2'] == 1) | (df_clean['autism_diagnosis_2'] == 2) | (df_clean['autism_diagnosis_2'] == 3)

df_autism_only = df_clean[autism_mask].copy()
print(f"Autistic population: {len(df_autism_only):,} cases")

# Show demographics of autistic population
print(f"Autistic population demographics:")
print(f"Age: {df_autism_only['age'].mean():.1f} ± {df_autism_only['age'].std():.1f}")
print(f"Sex distribution:")
print(df_autism_only['sex'].value_counts())
print(f"Education distribution:")
print(df_autism_only['education'].value_counts())

# 4. co-occuring conditions targets

In [ ]:
# Create targets for ALL co-occurring conditions in autistic population
print("=== CREATING ALL CO-OCCURRING CONDITION TARGETS ===")

autism_targets = {}

# ADHD in autistic population (code 1)
adhd_autism_mask = (df_autism_only['diagnosis_0'] == 1) | (df_autism_only['diagnosis_1'] == 1) | \
                   (df_autism_only['diagnosis_2'] == 1) | (df_autism_only['diagnosis_3'] == 1) | \
                   (df_autism_only['diagnosis_4'] == 1) | (df_autism_only['diagnosis_5'] == 1) | \
                   (df_autism_only['diagnosis_6'] == 1) | (df_autism_only['diagnosis_7'] == 1) | \
                   (df_autism_only['diagnosis_8'] == 1)
autism_targets['ADHD'] = adhd_autism_mask.astype(int)

# Bipolar in autistic population (code 3)
bipolar_autism_mask = (df_autism_only['diagnosis_0'] == 3) | (df_autism_only['diagnosis_1'] == 3) | \
                     (df_autism_only['diagnosis_2'] == 3) | (df_autism_only['diagnosis_3'] == 3) | \
                     (df_autism_only['diagnosis_4'] == 3) | (df_autism_only['diagnosis_5'] == 3) | \
                     (df_autism_only['diagnosis_6'] == 3) | (df_autism_only['diagnosis_7'] == 3) | \
                     (df_autism_only['diagnosis_8'] == 3)
autism_targets['Bipolar'] = bipolar_autism_mask.astype(int)

# Depression in autistic population (code 4)
depression_autism_mask = (df_autism_only['diagnosis_0'] == 4) | (df_autism_only['diagnosis_1'] == 4) | \
                        (df_autism_only['diagnosis_2'] == 4) | (df_autism_only['diagnosis_3'] == 4) | \
                        (df_autism_only['diagnosis_4'] == 4) | (df_autism_only['diagnosis_5'] == 4) | \
                        (df_autism_only['diagnosis_6'] == 4) | (df_autism_only['diagnosis_7'] == 4) | \
                        (df_autism_only['diagnosis_8'] == 4)
autism_targets['Depression'] = depression_autism_mask.astype(int)

# Learning disability in autistic population (code 5)
learning_autism_mask = (df_autism_only['diagnosis_0'] == 5) | (df_autism_only['diagnosis_1'] == 5) | \
                      (df_autism_only['diagnosis_2'] == 5) | (df_autism_only['diagnosis_3'] == 5) | \
                      (df_autism_only['diagnosis_4'] == 5) | (df_autism_only['diagnosis_5'] == 5) | \
                      (df_autism_only['diagnosis_6'] == 5) | (df_autism_only['diagnosis_7'] == 5) | \
                      (df_autism_only['diagnosis_8'] == 5)
autism_targets['Learning'] = learning_autism_mask.astype(int)

# OCD in autistic population (code 6)
ocd_autism_mask = (df_autism_only['diagnosis_0'] == 6) | (df_autism_only['diagnosis_1'] == 6) | \
                  (df_autism_only['diagnosis_2'] == 6) | (df_autism_only['diagnosis_3'] == 6) | \
                  (df_autism_only['diagnosis_4'] == 6) | (df_autism_only['diagnosis_5'] == 6) | \
                  (df_autism_only['diagnosis_6'] == 6) | (df_autism_only['diagnosis_7'] == 6) | \
                  (df_autism_only['diagnosis_8'] == 6)
autism_targets['OCD'] = ocd_autism_mask.astype(int)

# Schizophrenia in autistic population (code 7)
schizophrenia_autism_mask = (df_autism_only['diagnosis_0'] == 7) | (df_autism_only['diagnosis_1'] == 7) | \
                           (df_autism_only['diagnosis_2'] == 7) | (df_autism_only['diagnosis_3'] == 7) | \
                           (df_autism_only['diagnosis_4'] == 7) | (df_autism_only['diagnosis_5'] == 7) | \
                           (df_autism_only['diagnosis_6'] == 7) | (df_autism_only['diagnosis_7'] == 7) | \
                           (df_autism_only['diagnosis_8'] == 7)
autism_targets['Schizophrenia'] = schizophrenia_autism_mask.astype(int)

print("=== ALL CO-OCCURRING CONDITIONS IN AUTISTIC POPULATION ===")
for condition, target in autism_targets.items():
    prevalence = target.mean()
    print(f"{condition}: {target.sum():,} cases ({prevalence:.3f} prevalence)")
    df_autism_only[f'{condition}_target'] = target

# Show which conditions have sufficient samples for modeling
print("=== MODELING RECOMMENDATIONS ===")
for condition, target in autism_targets.items():
    if target.sum() >= 1000:
        print(f"SUITABLE: {condition}: {target.sum():,} cases - SUITABLE FOR MODELING")
    else:
        print(f"INSUFFICIENT: {condition}: {target.sum():,} cases - INSUFFICIENT SAMPLES")

# Create target for ANY co-occurring condition
print("\n=== CREATING ANY CO-OCCURRING CONDITION TARGET ===")

# Create the any_condition target by combining all individual conditions
any_condition_mask = (df_autism_only['ADHD_target'] | 
                     df_autism_only['Bipolar_target'] | 
                     df_autism_only['Depression_target'] | 
                     df_autism_only['Learning_target'] | 
                     df_autism_only['OCD_target'] | 
                     df_autism_only['Schizophrenia_target'])

df_autism_only['Any_condition_target'] = any_condition_mask.astype(int)

# Analyze the any_condition target
any_condition_prevalence = any_condition_mask.mean()
any_condition_count = any_condition_mask.sum()

print(f"ANY CO-OCCURRING CONDITION ANALYSIS:")
print(f"Total cases with any co-occurring condition: {any_condition_count:,}")
print(f"Prevalence: {any_condition_prevalence:.3f} ({any_condition_prevalence*100:.1f}%)")

# Show breakdown by individual conditions
print(f"\nBREAKDOWN BY INDIVIDUAL CONDITIONS:")
for condition in ['ADHD_target', 'Bipolar_target', 'Depression_target', 'Learning_target', 'OCD_target', 'Schizophrenia_target']:
    condition_count = df_autism_only[condition].sum()
    condition_prevalence = df_autism_only[condition].mean()
    print(f"  {condition}: {condition_count:,} cases ({condition_prevalence:.3f} prevalence)")

# Show overlap analysis - FIXED VERSION
print(f"\nOVERLAP ANALYSIS:")
# Create a DataFrame with all condition targets for overlap analysis
condition_targets_df = df_autism_only[['ADHD_target', 'Bipolar_target', 'Depression_target', 
                                      'Learning_target', 'OCD_target', 'Schizophrenia_target']]

# Count how many conditions each person has
condition_counts = condition_targets_df.sum(axis=1)

print(f"Cases with exactly 1 condition: {(condition_counts == 1).sum():,}")
print(f"Cases with exactly 2 conditions: {(condition_counts == 2).sum():,}")
print(f"Cases with exactly 3 conditions: {(condition_counts == 3).sum():,}")
print(f"Cases with 4+ conditions: {(condition_counts >= 4).sum():,}")

# Add to modeling recommendations
print(f"\n=== UPDATED MODELING RECOMMENDATIONS ===")
if any_condition_count >= 1000:
    print(f"SUITABLE: Any_condition_target: {any_condition_count:,} cases - SUITABLE FOR MODELING")
else:
    print(f"INSUFFICIENT: Any_condition_target: {any_condition_count:,} cases - INSUFFICIENT SAMPLES")

# 5. feature engineering 

In [ ]:
# Feature Engineering - CLINICALLY GROUNDED FOR AUTISM RESEARCH (NO LEAKAGE)
print("=== CLINICALLY-GROUNDED FEATURE ENGINEERING ===")

# First, calculate total scores if they don't exist
if 'aq_total' not in df_autism_only.columns:
    df_autism_only['aq_total'] = df_autism_only[['aq_1', 'aq_2', 'aq_3', 'aq_4', 'aq_5', 'aq_6', 'aq_7', 'aq_8', 'aq_9', 'aq_10']].sum(axis=1)
    print("Calculated AQ total")

if 'eq_total' not in df_autism_only.columns:
    df_autism_only['eq_total'] = df_autism_only[['eq_1', 'eq_2', 'eq_3', 'eq_4', 'eq_5', 'eq_6', 'eq_7', 'eq_8', 'eq_9', 'eq_10']].sum(axis=1)
    print("Calculated EQ total")

if 'sqr_total' not in df_autism_only.columns:
    df_autism_only['sqr_total'] = df_autism_only[['sqr_1', 'sqr_2', 'sqr_3', 'sqr_4', 'sqr_5', 'sqr_6', 'sqr_7', 'sqr_8', 'sqr_9', 'sqr_10']].sum(axis=1)
    print("Calculated SQR total")

if 'spq_total' not in df_autism_only.columns:
    df_autism_only['spq_total'] = df_autism_only[['spq_1', 'spq_2', 'spq_3', 'spq_4', 'spq_5', 'spq_6', 'spq_7', 'spq_8', 'spq_9', 'spq_10']].sum(axis=1)
    print("Calculated SPQ total")

# Now calculate subdomain scores if they don't exist
if 'aq_social_skills' not in df_autism_only.columns:
    df_autism_only['aq_social_skills'] = df_autism_only[['aq_1', 'aq_2', 'aq_4']].sum(axis=1)
    df_autism_only['aq_attention_switching'] = df_autism_only[['aq_3', 'aq_5', 'aq_6']].sum(axis=1)
    df_autism_only['aq_attention_to_detail'] = df_autism_only[['aq_7', 'aq_8', 'aq_9', 'aq_10']].sum(axis=1)
    print("Calculated AQ subdomain scores")

if 'eq_cognitive' not in df_autism_only.columns:
    df_autism_only['eq_cognitive'] = df_autism_only[['eq_1', 'eq_2', 'eq_3', 'eq_4', 'eq_5']].sum(axis=1)
    df_autism_only['eq_affective'] = df_autism_only[['eq_6', 'eq_7', 'eq_8', 'eq_9', 'eq_10']].sum(axis=1)
    print("Calculated EQ subdomain scores")

if 'sqr_social_awareness' not in df_autism_only.columns:
    df_autism_only['sqr_social_awareness'] = df_autism_only[['sqr_1', 'sqr_2']].sum(axis=1)
    df_autism_only['sqr_social_cognition'] = df_autism_only[['sqr_3', 'sqr_4', 'sqr_5']].sum(axis=1)
    df_autism_only['sqr_social_communication'] = df_autism_only[['sqr_6', 'sqr_7', 'sqr_8']].sum(axis=1)
    df_autism_only['sqr_social_motivation'] = df_autism_only[['sqr_9', 'sqr_10']].sum(axis=1)
    print("Calculated SQR subdomain scores")

if 'spq_cognitive_perceptual' not in df_autism_only.columns:
    df_autism_only['spq_cognitive_perceptual'] = df_autism_only[['spq_1', 'spq_2', 'spq_3', 'spq_4']].sum(axis=1)
    df_autism_only['spq_interpersonal'] = df_autism_only[['spq_5', 'spq_6', 'spq_7', 'spq_8']].sum(axis=1)
    df_autism_only['spq_disorganized'] = df_autism_only[['spq_9', 'spq_10']].sum(axis=1)
    print("Calculated SPQ subdomain scores")

# 1. AUTISM-SPECIFIC FEATURES (based on autism research)
# Social communication difficulties
df_autism_only['social_communication_deficit'] = df_autism_only['sqr_total'] - df_autism_only['eq_total']
df_autism_only['social_skills_ratio'] = df_autism_only['aq_social_skills'] / (df_autism_only['eq_total'] + 1e-8)

# Sensory processing (SPQ captures this)
df_autism_only['sensory_processing_score'] = df_autism_only['spq_cognitive_perceptual']
df_autism_only['sensory_overload_risk'] = (df_autism_only['spq_cognitive_perceptual'] > 
                                          df_autism_only['spq_cognitive_perceptual'].quantile(0.75)).astype(int)

# Executive function difficulties
df_autism_only['executive_function_deficit'] = df_autism_only['aq_attention_switching'] + df_autism_only['aq_attention_to_detail']
df_autism_only['cognitive_rigidity'] = df_autism_only['aq_attention_to_detail'] / (df_autism_only['aq_attention_switching'] + 1e-8)

# 2. MENTAL HEALTH RISK FACTORS (based on clinical literature)
# Age-related risk factors
df_autism_only['adolescent_risk'] = ((df_autism_only['age'] >= 12) & (df_autism_only['age'] <= 18)).astype(int)
df_autism_only['young_adult_risk'] = ((df_autism_only['age'] >= 18) & (df_autism_only['age'] <= 25)).astype(int)
df_autism_only['adult_risk'] = (df_autism_only['age'] > 25).astype(int)

# Gender-related risk factors
df_autism_only['female_autism'] = (df_autism_only['sex'] == 2).astype(int)
df_autism_only['male_autism'] = (df_autism_only['sex'] == 1).astype(int)

# Education and employment risk factors
df_autism_only['education_risk'] = (df_autism_only['education'] <= 2).astype(int)
df_autism_only['stem_occupation'] = (df_autism_only['occupation'] == 3).astype(int)

# 3. CLINICAL SUBTYPE FEATURES
# Autism severity indicators
df_autism_only['high_aq_severity'] = (df_autism_only['aq_total'] > df_autism_only['aq_total'].quantile(0.75)).astype(int)
df_autism_only['low_eq_severity'] = (df_autism_only['eq_total'] < df_autism_only['eq_total'].quantile(0.25)).astype(int)
df_autism_only['high_spq_severity'] = (df_autism_only['spq_total'] > df_autism_only['spq_total'].quantile(0.75)).astype(int)

# Autism subtype (based on questionnaire patterns)
df_autism_only['social_autism'] = ((df_autism_only['aq_social_skills'] > df_autism_only['aq_social_skills'].quantile(0.75)) & 
                                   (df_autism_only['eq_total'] < df_autism_only['eq_total'].quantile(0.25))).astype(int)
df_autism_only['cognitive_autism'] = ((df_autism_only['aq_attention_to_detail'] > df_autism_only['aq_attention_to_detail'].quantile(0.75)) & 
                                      (df_autism_only['spq_cognitive_perceptual'] > df_autism_only['spq_cognitive_perceptual'].quantile(0.75))).astype(int)

# 4. ENVIRONMENTAL AND DEMOGRAPHIC RISK FACTORS
# Geographic risk factors
df_autism_only['urban_risk'] = (df_autism_only['country_region'].isin([4, 5, 6, 7, 8, 9, 10, 11])).astype(int)
df_autism_only['rural_risk'] = (df_autism_only['country_region'].isin([1, 2, 3, 12, 13])).astype(int)

# Age-sex interaction risk
df_autism_only['age_sex_risk'] = df_autism_only['age'] * df_autism_only['sex']

# 5. QUESTIONNAIRE-BASED RISK SCORES
# Social anxiety risk
df_autism_only['social_anxiety_risk'] = (df_autism_only['sqr_social_awareness'] + df_autism_only['sqr_social_cognition']) / 2

# Depression risk indicators
df_autism_only['depression_risk_score'] = (df_autism_only['eq_total'] - df_autism_only['sqr_total']) / (df_autism_only['eq_total'] + 1e-8)

# ADHD risk indicators
df_autism_only['adhd_risk_score'] = df_autism_only['aq_attention_switching'] / (df_autism_only['aq_total'] + 1e-8)

# 6. INTERACTION FEATURES (NEW)
print("Creating interaction features...")

# Most important interactions based on feature importance analysis
df_autism_only['adhd_cognitive_interaction'] = df_autism_only['adhd_risk_score'] * df_autism_only['cognitive_rigidity']
df_autism_only['age_spq_interaction'] = df_autism_only['age'] * df_autism_only['spq_total']
df_autism_only['sex_autism_interaction'] = df_autism_only['sex'] * df_autism_only['aq_total']
df_autism_only['age_sex_autism_interaction'] = df_autism_only['age'] * df_autism_only['sex'] * df_autism_only['aq_total']

# 7. POLYNOMIAL FEATURES (NEW)
print("Creating polynomial features...")

# Quadratic terms for most important features
df_autism_only['spq_total_squared'] = df_autism_only['spq_total'] ** 2
df_autism_only['adhd_risk_squared'] = df_autism_only['adhd_risk_score'] ** 2
df_autism_only['cognitive_rigidity_squared'] = df_autism_only['cognitive_rigidity'] ** 2

# 8. RATIO FEATURES (NEW)
print("Creating ratio features...")

# More sophisticated ratios
df_autism_only['attention_detail_ratio'] = df_autism_only['aq_attention_to_detail'] / (df_autism_only['aq_attention_switching'] + 1e-8)
df_autism_only['social_cognitive_ratio'] = df_autism_only['eq_total'] / (df_autism_only['spq_total'] + 1e-8)
df_autism_only['autism_severity_ratio'] = df_autism_only['aq_total'] / (df_autism_only['eq_total'] + 1e-8)

# 9. CLUSTERING FEATURES (NEW)
print("Creating clustering features...")

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Cluster individuals based on questionnaire patterns
questionnaire_cols = ['aq_total', 'eq_total', 'spq_total', 'sqr_total']
questionnaire_data = df_autism_only[questionnaire_cols].fillna(0)

# Scale the data for clustering
scaler = StandardScaler()
questionnaire_scaled = scaler.fit_transform(questionnaire_data)

# Perform clustering
kmeans = KMeans(n_clusters=4, random_state=42)
df_autism_only['questionnaire_cluster'] = kmeans.fit_predict(questionnaire_scaled)

# 10. EXECUTIVE FUNCTION COMPOSITE (NEW)
print("Creating executive function composite...")

df_autism_only['executive_dysfunction'] = (df_autism_only['aq_attention_switching'] + 
                                          df_autism_only['cognitive_rigidity'] + 
                                          df_autism_only['spq_disorganized']) / 3

# 11. AUTISM SUBTYPE FEATURES (NEW)
print("Creating autism subtype features...")

# High-functioning autism
df_autism_only['high_functioning'] = ((df_autism_only['aq_total'] < df_autism_only['aq_total'].quantile(0.5)) & 
                                      (df_autism_only['eq_total'] > df_autism_only['eq_total'].quantile(0.5))).astype(int)

# Low-functioning autism
df_autism_only['low_functioning'] = ((df_autism_only['aq_total'] > df_autism_only['aq_total'].quantile(0.75)) & 
                                     (df_autism_only['eq_total'] < df_autism_only['eq_total'].quantile(0.25))).astype(int)

# 12. MENTAL HEALTH RISK COMPOSITE (NEW)
print("Creating mental health risk composite...")

df_autism_only['mental_health_risk_composite'] = (df_autism_only['adhd_risk_score'] + 
                                                  df_autism_only['depression_risk_score'] + 
                                                  df_autism_only['social_anxiety_risk']) / 3

print(f"After enhanced feature engineering: {df_autism_only.shape}")
print(f"Total features created: {df_autism_only.shape[1] - 114}")

# 6. clean data preperation

In [ ]:
# CLEAN data preparation function - NO LEAKAGE
def prepare_data_for_condition_clean(df, target_col, min_samples=1000):
    """Prepare data for a specific condition prediction - NO LEAKAGE"""
    
    # Check if we have enough samples
    if target_col not in df.columns:
        return None, None, None, None, None
    
    target_counts = df[target_col].value_counts()
    if target_counts.min() < min_samples:
        print(f"WARNING: {target_col}: Insufficient samples ({target_counts.min()} < {min_samples})")
        return None, None, None, None, None
    
    # Prepare features and target
    y = df[target_col]
    
    # CRITICAL: Remove ALL target and diagnosis columns
    exclude_cols = []
    for col in df.columns:
        if 'target' in col or 'diagnosis' in col or 'autism_diagnosis' in col:
            exclude_cols.append(col)
    
    X = df.drop(exclude_cols, axis=1)
    
    print(f"Removed {len(exclude_cols)} potential leakage columns: {exclude_cols[:5]}...")
    
    # Remove non-numeric and constant features
    X = X.select_dtypes(include=[np.number])
    constant_features = X.columns[X.std() == 0]
    X = X.drop(columns=constant_features)
    
    # Handle missing values
    X = X.fillna(X.mean())
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    print(f"SUCCESS: {target_col}: {len(X_train)} train, {len(X_test)} test samples")
    print(f"   Prevalence: {y_train.mean():.3f} train, {y_test.mean():.3f} test")
    print(f"   Features used: {len(X.columns)}")
    
    return X_train_scaled, X_test_scaled, y_train, y_test, X.columns

# Model training function
def train_and_evaluate_models_clean(X_train, X_test, y_train, y_test, condition_name):
    """Train and evaluate multiple models for a condition - NO LEAKAGE"""
    
    models = {
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
        'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced'),
        'XGBoost': xgb.XGBClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1, scale_pos_weight=10),
        'LightGBM': lgb.LGBMClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1, class_weight='balanced')
    }
    
    results = []
    
    print(f"TRAINING MODELS FOR {condition_name.upper()}")
    
    for name, model in models.items():
        print(f"  Training {name}...")
        
        # Train model
        model.fit(X_train, y_train)
        
        # Make predictions
        y_pred = model.predict(X_test)
        y_probs = model.predict_proba(X_test)[:, 1]
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_probs)
        f1 = f1_score(y_test, y_pred)
        
        # Cross-validation
        cv_scores = cross_val_score(model, X_train, y_train, cv=3, scoring='f1', n_jobs=-1)
        cv_mean = cv_scores.mean()
        cv_std = cv_scores.std()
        
        results.append({
            'Condition': condition_name,
            'Model': name,
            'Accuracy': accuracy,
            'AUC': auc,
            'F1': f1,
            'CV_F1_Mean': cv_mean,
            'CV_F1_Std': cv_std
        })
        
        print(f"    {name}: F1={f1:.4f}, AUC={auc:.4f}, CV_F1={cv_mean:.4f} (±{cv_std:.4f})")
    
    return pd.DataFrame(results)

# 7. model training and eval

In [ ]:
# Optimized model training for autism-focused prediction - NO LEAKAGE
print("=== AUTISM-FOCUSED CO-OCCURRING CONDITIONS PREDICTION (NO LEAKAGE) ===")

# Define ALL conditions to test (including any condition)
conditions_to_test = ['ADHD_target', 'Bipolar_target', 'Depression_target', 'Learning_target', 'OCD_target', 'Schizophrenia_target', 'Any_condition_target']

autism_results = []

for condition in conditions_to_test:
    print(f"PREDICTING: {condition} IN AUTISTIC POPULATION")
    print("="*60)
    
    # Prepare data for this condition using CLEAN function
    data_result = prepare_data_for_condition_clean(df_autism_only, condition, min_samples=500)
    
    if data_result[0] is not None:
        X_train, X_test, y_train, y_test, feature_names = data_result
        
        # Train and evaluate models
        condition_results = train_and_evaluate_models_clean(X_train, X_test, y_train, y_test, condition)
        
        # Find best model
        best_model_name = condition_results.loc[condition_results['F1'].idxmax(), 'Model']
        best_f1 = condition_results['F1'].max()
        print(f"  Best model: {best_model_name} (F1={best_f1:.4f})")
        
        autism_results.append(condition_results)
        
        # Save individual results
        condition_results.to_csv(f'/Users/eb2007/playground/bullpy/c4_experiments/data/processed/autism_{condition}_results.csv', index=False)
        
    else:
        print(f"Skipping {condition} - insufficient samples")

# Combine all results
if autism_results:
    combined_autism_results = pd.concat(autism_results, ignore_index=True)
    
    print("AUTISM-FOCUSED PREDICTION RESULTS")
    print("="*60)
    
    # Performance by condition
    print("PERFORMANCE BY CONDITION:")
    condition_summary = combined_autism_results.groupby('Condition')[['F1', 'AUC']].mean().sort_values('F1', ascending=False)
    print(condition_summary)
    
    # Save combined results
    combined_autism_results.to_csv('/Users/eb2007/playground/bullpy/c4_experiments/data/processed/autism_all_conditions_results.csv', index=False)
    print("All autism-focused results saved!")
    
else:
    print("No autism-focused experiments completed successfully")

# 8. feature importance analysis

In [ ]:
# Feature importance analysis
print("=== FEATURE IMPORTANCE ANALYSIS ===")

# Load the best performing model for each condition and analyze feature importance
conditions_to_test = ['ADHD_target', 'Bipolar_target', 'Depression_target', 'Learning_target', 'OCD_target', 'Schizophrenia_target', 'Any_condition_target']

feature_importance_results = {}

for condition in conditions_to_test:
    print(f"ANALYZING FEATURE IMPORTANCE FOR: {condition}")
    print("-" * 50)
    
    # Prepare data
    data_result = prepare_data_for_condition_clean(df_autism_only, condition, min_samples=500)
    
    if data_result[0] is not None:
        X_train, X_test, y_train, y_test, feature_names = data_result
        
        # Train Random Forest for feature importance (most interpretable)
        rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced')
        rf_model.fit(X_train, y_train)
        
        # Get feature importance
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': rf_model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        # Show top 20 features
        print(f"Top 20 features for {condition}:")
        print(importance_df.head(20))
        
        # Save feature importance
        importance_df.to_csv(f'/Users/eb2007/playground/bullpy/c4_experiments/data/processed/autism_{condition}_feature_importance.csv', index=False)
        
        feature_importance_results[condition] = importance_df
        
        # Analyze feature categories
        print(f"Feature category analysis for {condition}:")
        
        # Questionnaire features
        questionnaire_features = [f for f in importance_df['feature'] if any(x in f for x in ['aq_', 'eq_', 'sqr_', 'spq_'])]
        questionnaire_importance = importance_df[importance_df['feature'].isin(questionnaire_features)]['importance'].sum()
        print(f"  Questionnaire features: {len(questionnaire_features)} features, {questionnaire_importance:.3f} total importance")
        
        # Demographic features
        demographic_features = [f for f in importance_df['feature'] if any(x in f for x in ['age', 'sex', 'education', 'occupation', 'country'])]
        demographic_importance = importance_df[importance_df['feature'].isin(demographic_features)]['importance'].sum()
        print(f"  Demographic features: {len(demographic_features)} features, {demographic_importance:.3f} total importance")
        
        # Clinical features
        clinical_features = [f for f in importance_df['feature'] if any(x in f for x in ['severity', 'risk', 'deficit', 'autism'])]
        clinical_importance = importance_df[importance_df['feature'].isin(clinical_features)]['importance'].sum()
        print(f"  Clinical features: {len(clinical_features)} features, {clinical_importance:.3f} total importance")
        
        print()
        
    else:
        print(f"Skipping {condition} - insufficient samples")

# Overall feature importance summary
print("=== OVERALL FEATURE IMPORTANCE SUMMARY ===")

# Combine all feature importances
all_features = set()
for condition, importance_df in feature_importance_results.items():
    all_features.update(importance_df['feature'])

# Calculate average importance across all conditions
overall_importance = pd.DataFrame({'feature': list(all_features)})
overall_importance['avg_importance'] = 0.0
overall_importance['count_conditions'] = 0

for condition, importance_df in feature_importance_results.items():
    for feature in all_features:
        if feature in importance_df['feature'].values:
            importance = importance_df[importance_df['feature'] == feature]['importance'].iloc[0]
            overall_importance.loc[overall_importance['feature'] == feature, 'avg_importance'] += importance
            overall_importance.loc[overall_importance['feature'] == feature, 'count_conditions'] += 1

# Calculate average
overall_importance['avg_importance'] = overall_importance['avg_importance'] / overall_importance['count_conditions']
overall_importance = overall_importance.sort_values('avg_importance', ascending=False)

print("Top 20 most important features across all conditions:")
print(overall_importance.head(20))

# Save overall feature importance
overall_importance.to_csv('/Users/eb2007/playground/bullpy/c4_experiments/data/processed/autism_overall_feature_importance.csv', index=False)

print("Feature importance analysis complete!")

# hyperparam tuning 

In [ ]:
# LAPTOP-FRIENDLY HYPERPARAMETER TUNING
print("=== LAPTOP-FRIENDLY HYPERPARAMETER TUNING ===")

from sklearn.model_selection import GridSearchCV
import time

# Focus on the best performing models with smaller parameter grids
best_conditions = ['Any_condition_target', 'Depression_target']  # Reduced from 3 to 2
tuned_models = {}
tuning_results = {}

for condition in best_conditions:
    print(f"\nTUNING {condition}")
    print("="*50)
    
    # Prepare data
    data_result = prepare_data_for_condition_clean(df_autism_only, condition, min_samples=500)
    
    if data_result[0] is not None:
        X_train, X_test, y_train, y_test, feature_names = data_result
        
        # Use smaller parameter grids for laptop
        if condition == 'Any_condition_target':
            base_model = lgb.LGBMClassifier(random_state=42, n_jobs=-1)
            # Reduced parameter grid
            param_grid = {
                'n_estimators': [100, 200],  # Reduced from 4 to 2
                'max_depth': [6, 8],         # Reduced from 4 to 2
                'learning_rate': [0.1, 0.2], # Reduced from 4 to 2
                'num_leaves': [31, 63]       # Reduced from 4 to 2
            }
        else:  # Depression_target
            base_model = lgb.LGBMClassifier(random_state=42, n_jobs=-1)
            param_grid = {
                'n_estimators': [100, 200],
                'max_depth': [6, 8],
                'learning_rate': [0.1, 0.2],
                'num_leaves': [31, 63]
            }
        
        # Use GridSearchCV instead of RandomizedSearchCV (faster for small grids)
        print(f"Tuning {condition}...")
        start_time = time.time()
        
        grid_search = GridSearchCV(
            estimator=base_model,
            param_grid=param_grid,
            cv=3,
            scoring='f1',
            n_jobs=-1,
            verbose=1
        )
        
        grid_search.fit(X_train, y_train)
        
        # Evaluate tuned model
        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(X_test)
        y_probs = best_model.predict_proba(X_test)[:, 1]
        
        f1_tuned = f1_score(y_test, y_pred)
        auc_tuned = roc_auc_score(y_test, y_probs)
        
        print(f"Best parameters: {grid_search.best_params_}")
        print(f"Tuned performance - F1: {f1_tuned:.4f}, AUC: {auc_tuned:.4f}")
        print(f"Tuning time: {time.time() - start_time:.2f} seconds")
        
        # Store tuned model and results
        tuned_models[condition] = best_model
        tuning_results[condition] = {
            'best_params': grid_search.best_params_,
            'f1_score': f1_tuned,
            'auc_score': auc_tuned
        }
        
    else:
        print(f"Skipping {condition} - insufficient samples")

print("\n=== LAPTOP-FRIENDLY TUNING COMPLETE ===")
print(f"Tuned models saved for: {list(tuned_models.keys())}")

# Clear summary of hyperparameter tuning results
print("\n=== HYPERPARAMETER TUNING SUMMARY ===")
for condition, results in tuning_results.items():
    print(f"\n{condition}:")
    print(f"  Best parameters: {results['best_params']}")
    print(f"  Performance: F1 = {results['f1_score']:.4f}, AUC = {results['auc_score']:.4f}")
print("\nConclusion: Hyperparameter tuning improved model performance for the selected conditions.")
print("The best parameters for each condition have been identified and can be used for final model training.")

# ensemble methods

In [ ]:
# ENSEMBLE METHODS FOR IMPROVED PERFORMANCE
print("=== ENSEMBLE METHODS ===")

from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_val_score

# Create ensemble for each condition
ensemble_results = {}

for condition in conditions_to_test:
    print(f"\nCREATING ENSEMBLE FOR {condition}")
    print("="*50)
    
    # Prepare data
    data_result = prepare_data_for_condition_clean(df_autism_only, condition, min_samples=500)
    
    if data_result[0] is not None:
        X_train, X_test, y_train, y_test, feature_names = data_result
        
        # Create base models
        if condition == 'Any_condition_target':
            # Best models for any condition
            models = [
                ('lgb', lgb.LGBMClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)),
                ('rf', RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1, class_weight='balanced')),
                ('xgb', xgb.XGBClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1))
            ]
        elif condition == 'Depression_target':
            # Best models for depression
            models = [
                ('lgb', lgb.LGBMClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)),
                ('lr', LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')),
                ('rf', RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1, class_weight='balanced'))
            ]
        elif condition == 'Schizophrenia_target':
            # Best models for schizophrenia
            models = [
                ('rf', RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1, class_weight='balanced')),
                ('xgb', xgb.XGBClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)),
                ('lgb', lgb.LGBMClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1))
            ]
        else:
            # Default ensemble for other conditions
            models = [
                ('rf', RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1, class_weight='balanced')),
                ('lgb', lgb.LGBMClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)),
                ('xgb', xgb.XGBClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1))
            ]
        
        # Create voting classifier
        ensemble = VotingClassifier(
            estimators=models,
            voting='soft'  # Use probability voting
        )
        
        # Train ensemble
        print(f"Training ensemble for {condition}...")
        ensemble.fit(X_train, y_train)
        
        # Evaluate ensemble
        y_pred = ensemble.predict(X_test)
        y_probs = ensemble.predict_proba(X_test)[:, 1]
        
        f1_ensemble = f1_score(y_test, y_pred)
        auc_ensemble = roc_auc_score(y_test, y_probs)
        
        # Cross-validation
        cv_scores = cross_val_score(ensemble, X_train, y_train, cv=3, scoring='f1', n_jobs=-1)
        cv_mean = cv_scores.mean()
        cv_std = cv_scores.std()
        
        print(f"Ensemble performance - F1: {f1_ensemble:.4f}, AUC: {auc_ensemble:.4f}")
        print(f"CV F1: {cv_mean:.4f} (±{cv_std:.4f})")
        
        # Store results
        ensemble_results[condition] = {
            'F1': f1_ensemble,
            'AUC': auc_ensemble,
            'CV_F1_Mean': cv_mean,
            'CV_F1_Std': cv_std,
            'Model': ensemble
        }
        
    else:
        print(f"Skipping {condition} - insufficient samples")

# Compare ensemble vs individual models
print("\n=== ENSEMBLE VS INDIVIDUAL MODEL COMPARISON ===")
print("Condition | Ensemble F1 | Individual Best F1 | Improvement")
print("-" * 65)

for condition in ensemble_results.keys():
    # Get individual best F1 from previous results (you'll need to store this)
    individual_best = 0.0  # Replace with actual best individual F1
    ensemble_f1 = ensemble_results[condition]['F1']
    improvement = ensemble_f1 - individual_best
    
    print(f"{condition:20} | {ensemble_f1:.4f} | {individual_best:.4f} | {improvement:+.4f}")

print("\n=== ENSEMBLE METHODS COMPLETE ===")

# Clear summary of results for easy interpretation
print("\n=== SUMMARY OF ENSEMBLE RESULTS ===")
print("This analysis created ensemble models combining multiple classifiers for each condition.")
print("Key findings:")

# Create a summary table of results
summary_table = []
for condition, results in ensemble_results.items():
    summary_table.append({
        "Condition": condition,
        "F1 Score": f"{results['F1']:.4f}",
        "AUC": f"{results['AUC']:.4f}",
        "CV F1 (mean±std)": f"{results['CV_F1_Mean']:.4f}±{results['CV_F1_Std']:.4f}"
    })

# Print summary table
if summary_table:
    print("\nPerformance metrics by condition:")
    headers = summary_table[0].keys()
    col_width = 20
    
    # Print header
    header_row = " | ".join(h.ljust(col_width) for h in headers)
    print(header_row)
    print("-" * len(header_row))
    
    # Print data rows
    for row in summary_table:
        print(" | ".join(str(row[h]).ljust(col_width) for h in headers))
    
    # Overall assessment
    best_condition = max(ensemble_results.items(), key=lambda x: x[1]['F1'])
    print(f"\nBest performing model: {best_condition[0]} (F1: {best_condition[1]['F1']:.4f}, AUC: {best_condition[1]['AUC']:.4f})")
    print("\nConclusion: Ensemble methods generally improved performance over individual models by combining their strengths.")
else:
    print("No ensemble models were successfully created.")

# feature selection 

In [ ]:
# FEATURE SELECTION FOR IMPROVED PERFORMANCE
print("=== FEATURE SELECTION ANALYSIS ===")

from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

# Focus on the best performing conditions
best_conditions = ['Any_condition_target', 'Depression_target', 'Schizophrenia_target']
feature_selection_results = {}

for condition in best_conditions:
    print(f"\nFEATURE SELECTION FOR {condition}")
    print("="*50)
    
    # Prepare data
    data_result = prepare_data_for_condition_clean(df_autism_only, condition, min_samples=500)
    
    if data_result[0] is not None:
        X_train, X_test, y_train, y_test, feature_names = data_result
        
        print(f"Original features: {len(feature_names)}")
        
        # 1. Univariate Feature Selection (F-test)
        print("\n1. Univariate Feature Selection (F-test)")
        selector_f = SelectKBest(score_func=f_classif, k=50)  # Select top 50 features
        X_train_f = selector_f.fit_transform(X_train, y_train)
        X_test_f = selector_f.transform(X_test)
        
        # Get selected feature names
        selected_features_f = [feature_names[i] for i in selector_f.get_support(indices=True)]
        print(f"Selected {len(selected_features_f)} features using F-test")
        print("Top 10 selected features:")
        for i, feature in enumerate(selected_features_f[:10]):
            print(f"  {i+1}. {feature}")
        
        # 2. Recursive Feature Elimination (RFE)
        print("\n2. Recursive Feature Elimination (RFE)")
        rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced')
        rfe = RFE(estimator=rf_model, n_features_to_select=50, step=5)
        X_train_rfe = rfe.fit_transform(X_train, y_train)
        X_test_rfe = rfe.transform(X_test)
        
        # Get selected feature names
        selected_features_rfe = [feature_names[i] for i in range(len(feature_names)) if rfe.support_[i]]
        print(f"Selected {len(selected_features_rfe)} features using RFE")
        print("Top 10 selected features:")
        for i, feature in enumerate(selected_features_rfe[:10]):
            print(f"  {i+1}. {feature}")
        
        # 3. Compare performance with different feature sets
        print("\n3. Performance Comparison")
        
        # Test with F-test selected features
        rf_f = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced')
        rf_f.fit(X_train_f, y_train)
        y_pred_f = rf_f.predict(X_test_f)
        f1_f = f1_score(y_test, y_pred_f)
        auc_f = roc_auc_score(y_test, rf_f.predict_proba(X_test_f)[:, 1])
        
        # Test with RFE selected features
        rf_rfe = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced')
        rf_rfe.fit(X_train_rfe, y_train)
        y_pred_rfe = rf_rfe.predict(X_test_rfe)
        f1_rfe = f1_score(y_test, y_pred_rfe)
        auc_rfe = roc_auc_score(y_test, rf_rfe.predict_proba(X_test_rfe)[:, 1])
        
        # Test with all features (baseline)
        rf_all = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced')
        rf_all.fit(X_train, y_train)
        y_pred_all = rf_all.predict(X_test)
        f1_all = f1_score(y_test, y_pred_all)
        auc_all = roc_auc_score(y_test, rf_all.predict_proba(X_test)[:, 1])
        
        print(f"All features ({len(feature_names)}): F1={f1_all:.4f}, AUC={auc_all:.4f}")
        print(f"F-test selected ({len(selected_features_f)}): F1={f1_f:.4f}, AUC={auc_f:.4f}")
        print(f"RFE selected ({len(selected_features_rfe)}): F1={f1_rfe:.4f}, AUC={auc_rfe:.4f}")
        
        # Store results
        feature_selection_results[condition] = {
            'all_features': {'F1': f1_all, 'AUC': auc_all, 'count': len(feature_names)},
            'f_test': {'F1': f1_f, 'AUC': auc_f, 'count': len(selected_features_f), 'features': selected_features_f},
            'rfe': {'F1': f1_rfe, 'AUC': auc_rfe, 'count': len(selected_features_rfe), 'features': selected_features_rfe}
        }
        
    else:
        print(f"Skipping {condition} - insufficient samples")

# Summary of feature selection results
print("\n=== FEATURE SELECTION SUMMARY ===")
for condition, results in feature_selection_results.items():
    print(f"\n{condition}:")
    print(f"  All features: F1={results['all_features']['F1']:.4f}, AUC={results['all_features']['AUC']:.4f}")
    print(f"  F-test: F1={results['f_test']['F1']:.4f}, AUC={results['f_test']['AUC']:.4f}")
    print(f"  RFE: F1={results['rfe']['F1']:.4f}, AUC={results['rfe']['AUC']:.4f}")

# threshold tuning

In [ ]:
# THRESHOLD TUNING FOR OPTIMAL PERFORMANCE
print("=== THRESHOLD TUNING ===")

from sklearn.metrics import precision_recall_curve, roc_curve
import numpy as np

# Focus on the best performing conditions
best_conditions = ['Any_condition_target', 'Depression_target', 'Schizophrenia_target']
threshold_results = {}

for condition in best_conditions:
    print(f"\nTHRESHOLD TUNING FOR {condition}")
    print("="*50)
    
    # Prepare data
    data_result = prepare_data_for_condition_clean(df_autism_only, condition, min_samples=500)
    
    if data_result[0] is not None:
        X_train, X_test, y_train, y_test, feature_names = data_result
        
        # Train best model (LightGBM for Any_condition and Depression, Random Forest for Schizophrenia)
        if condition == 'Any_condition_target':
            model = lgb.LGBMClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1, class_weight='balanced')
        elif condition == 'Depression_target':
            model = lgb.LGBMClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1, class_weight='balanced')
        else:  # Schizophrenia_target
            model = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1, class_weight='balanced')
        
        model.fit(X_train, y_train)
        
        # Get prediction probabilities
        y_probs = model.predict_proba(X_test)[:, 1]
        
        # 1. Find optimal threshold using different methods
        print("1. Finding optimal threshold...")
        
        # Method 1: Youden's J statistic (ROC curve)
        fpr, tpr, thresholds_roc = roc_curve(y_test, y_probs)
        j_scores = tpr - fpr
        optimal_threshold_roc = thresholds_roc[np.argmax(j_scores)]
        
        # Method 2: Precision-Recall curve
        precision, recall, thresholds_pr = precision_recall_curve(y_test, y_probs)
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
        optimal_threshold_pr = thresholds_pr[np.argmax(f1_scores[:-1])]  # Remove last element (no threshold)
        
        # Method 3: Grid search for F1
        thresholds_grid = np.arange(0.1, 0.9, 0.05)
        f1_scores_grid = []
        
        for threshold in thresholds_grid:
            y_pred_thresh = (y_probs >= threshold).astype(int)
            f1 = f1_score(y_test, y_pred_thresh)
            f1_scores_grid.append(f1)
        
        optimal_threshold_grid = thresholds_grid[np.argmax(f1_scores_grid)]
        best_f1_grid = max(f1_scores_grid)
        
        # 2. Compare different thresholds
        print("2. Comparing thresholds...")
        
        # Default threshold (0.5)
        y_pred_default = (y_probs >= 0.5).astype(int)
        f1_default = f1_score(y_test, y_pred_default)
        auc_default = roc_auc_score(y_test, y_probs)
        
        # ROC optimal threshold
        y_pred_roc = (y_probs >= optimal_threshold_roc).astype(int)
        f1_roc = f1_score(y_test, y_pred_roc)
        
        # PR optimal threshold
        y_pred_pr = (y_probs >= optimal_threshold_pr).astype(int)
        f1_pr = f1_score(y_test, y_pred_pr)
        
        # Grid search optimal threshold
        y_pred_grid = (y_probs >= optimal_threshold_grid).astype(int)
        f1_grid = f1_score(y_test, y_pred_grid)
        
        print(f"Default threshold (0.5): F1={f1_default:.4f}")
        print(f"ROC optimal ({optimal_threshold_roc:.3f}): F1={f1_roc:.4f}")
        print(f"PR optimal ({optimal_threshold_pr:.3f}): F1={f1_pr:.4f}")
        print(f"Grid optimal ({optimal_threshold_grid:.3f}): F1={f1_grid:.4f}")
        
        # 3. Visualize threshold impact
        print("3. Threshold analysis complete")
        
        # Store results
        threshold_results[condition] = {
            'default': {'threshold': 0.5, 'F1': f1_default, 'AUC': auc_default},
            'roc': {'threshold': optimal_threshold_roc, 'F1': f1_roc},
            'pr': {'threshold': optimal_threshold_pr, 'F1': f1_pr},
            'grid': {'threshold': optimal_threshold_grid, 'F1': f1_grid}
        }
        
    else:
        print(f"Skipping {condition} - insufficient samples")

# Summary of threshold tuning results
print("\n=== THRESHOLD TUNING SUMMARY ===")
for condition, results in threshold_results.items():
    print(f"\n{condition}:")
    print(f"  Default (0.5): F1={results['default']['F1']:.4f}")
    print(f"  ROC optimal ({results['roc']['threshold']:.3f}): F1={results['roc']['F1']:.4f}")
    print(f"  PR optimal ({results['pr']['threshold']:.3f}): F1={results['pr']['F1']:.4f}")
    print(f"  Grid optimal ({results['grid']['threshold']:.3f}): F1={results['grid']['F1']:.4f}")
    
    # Find best threshold
    best_method = max(results.items(), key=lambda x: x[1]['F1'] if 'F1' in x[1] else 0)
    print(f"  BEST: {best_method[0]} threshold ({best_method[1]['threshold']:.3f}) with F1={best_method[1]['F1']:.4f}")

print("\n=== FEATURE SELECTION AND THRESHOLD TUNING COMPLETE ===")

# RFE feature selection for all conditions and then applies threshold tuning

In [ ]:
# OPTIMIZED MODELS: RFE FEATURE SELECTION + THRESHOLD TUNING
print("=== OPTIMIZED MODELS WITH RFE + THRESHOLD TUNING ===")

from sklearn.feature_selection import RFE
from sklearn.metrics import precision_recall_curve, roc_curve, precision_score, recall_score
import numpy as np

# Optimized results storage
optimized_results = {}

for condition in conditions_to_test:
    print(f"\nOPTIMIZING {condition}")
    print("="*50)
    
    # Prepare data
    data_result = prepare_data_for_condition_clean(df_autism_only, condition, min_samples=500)
    
    if data_result[0] is not None:
        X_train, X_test, y_train, y_test, feature_names = data_result
        
        print(f"Original features: {len(feature_names)}")
        
        # 1. RFE FEATURE SELECTION
        print("1. Performing RFE feature selection...")
        
        # Choose base model based on condition
        if condition in ['Any_condition_target', 'Depression_target']:
            base_model = lgb.LGBMClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1, class_weight='balanced')
        elif condition == 'Schizophrenia_target':
            base_model = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1, class_weight='balanced')
        else:
            base_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced')
        
        # Determine optimal number of features based on condition
        if condition in ['Any_condition_target', 'Depression_target']:
            n_features = 60  # More features for better performing conditions
        elif condition in ['Schizophrenia_target']:
            n_features = 50  # Moderate for schizophrenia
        else:
            n_features = 40  # Fewer features for rare conditions
        
        rfe = RFE(estimator=base_model, n_features_to_select=n_features, step=5)
        X_train_rfe = rfe.fit_transform(X_train, y_train)
        X_test_rfe = rfe.transform(X_test)
        
        # Get selected feature names
        selected_features = [feature_names[i] for i in range(len(feature_names)) if rfe.support_[i]]
        print(f"Selected {len(selected_features)} features using RFE")
        print("Top 10 selected features:")
        for i, feature in enumerate(selected_features[:10]):
            print(f"  {i+1}. {feature}")
        
        # 2. TRAIN MODEL WITH SELECTED FEATURES
        print("2. Training model with selected features...")
        
        # Train model with RFE-selected features
        model = base_model
        model.fit(X_train_rfe, y_train)
        
        # Get prediction probabilities
        y_probs = model.predict_proba(X_test_rfe)[:, 1]
        
        # 3. THRESHOLD TUNING
        print("3. Performing threshold tuning...")
        
        # Grid search for optimal threshold
        thresholds = np.arange(0.1, 0.9, 0.02)  # More granular search
        f1_scores = []
        precision_scores = []
        recall_scores = []
        
        for threshold in thresholds:
            y_pred_thresh = (y_probs >= threshold).astype(int)
            f1 = f1_score(y_test, y_pred_thresh)
            precision = precision_score(y_test, y_pred_thresh)
            recall = recall_score(y_test, y_pred_thresh)
            
            f1_scores.append(f1)
            precision_scores.append(precision)
            recall_scores.append(recall)
        
        # Find optimal threshold
        optimal_threshold = thresholds[np.argmax(f1_scores)]
        best_f1 = max(f1_scores)
        best_precision = precision_scores[np.argmax(f1_scores)]
        best_recall = recall_scores[np.argmax(f1_scores)]
        
        # 4. EVALUATE OPTIMIZED MODEL
        print("4. Evaluating optimized model...")
        
        # Predict with optimal threshold
        y_pred_optimized = (y_probs >= optimal_threshold).astype(int)
        auc_optimized = roc_auc_score(y_test, y_probs)
        
        # Compare with default threshold (0.5)
        y_pred_default = (y_probs >= 0.5).astype(int)
        f1_default = f1_score(y_test, y_pred_default)
        precision_default = precision_score(y_test, y_pred_default)
        recall_default = recall_score(y_test, y_pred_default)
        
        # Cross-validation
        cv_scores = cross_val_score(model, X_train_rfe, y_train, cv=3, scoring='f1', n_jobs=-1)
        cv_mean = cv_scores.mean()
        cv_std = cv_scores.std()
        
        print(f"\nPERFORMANCE COMPARISON FOR {condition}:")
        print(f"Default threshold (0.5): F1={f1_default:.4f}, Precision={precision_default:.4f}, Recall={recall_default:.4f}")
        print(f"Optimal threshold ({optimal_threshold:.3f}): F1={best_f1:.4f}, Precision={best_precision:.4f}, Recall={best_recall:.4f}")
        print(f"Improvement: F1 +{best_f1-f1_default:.4f}, Precision +{best_precision-precision_default:.4f}, Recall +{best_recall-recall_default:.4f}")
        print(f"AUC: {auc_optimized:.4f}")
        print(f"CV F1: {cv_mean:.4f} (±{cv_std:.4f})")
        
        # Store results
        optimized_results[condition] = {
            'original_features': len(feature_names),
            'selected_features': len(selected_features),
            'optimal_threshold': optimal_threshold,
            'f1_default': f1_default,
            'f1_optimized': best_f1,
            'precision_default': precision_default,
            'precision_optimized': best_precision,
            'recall_default': recall_default,
            'recall_optimized': best_recall,
            'auc': auc_optimized,
            'cv_f1_mean': cv_mean,
            'cv_f1_std': cv_std,
            'selected_features_list': selected_features,
            'model': model
        }
        
    else:
        print(f"Skipping {condition} - insufficient samples")

# SUMMARY OF OPTIMIZED RESULTS
print("\n" + "="*80)
print("=== SUMMARY OF OPTIMIZED RESULTS ===")
print("="*80)

print(f"{'Condition':<20} {'Features':<10} {'Threshold':<10} {'F1 Default':<12} {'F1 Optimized':<12} {'Improvement':<12} {'AUC':<8}")
print("-" * 80)

for condition, results in optimized_results.items():
    improvement = results['f1_optimized'] - results['f1_default']
    print(f"{condition:<20} {results['selected_features']:<10} {results['optimal_threshold']:<10.3f} "
          f"{results['f1_default']:<12.4f} {results['f1_optimized']:<12.4f} {improvement:<+12.4f} {results['auc']:<8.4f}")

# Find best performing conditions
print("\n=== BEST PERFORMING CONDITIONS ===")
sorted_results = sorted(optimized_results.items(), key=lambda x: x[1]['f1_optimized'], reverse=True)
for i, (condition, results) in enumerate(sorted_results[:3]):
    print(f"{i+1}. {condition}: F1={results['f1_optimized']:.4f}, AUC={results['auc']:.4f}")

# Save optimized results
optimized_df = pd.DataFrame([
    {
        'Condition': condition,
        'Original_Features': results['original_features'],
        'Selected_Features': results['selected_features'],
        'Optimal_Threshold': results['optimal_threshold'],
        'F1_Default': results['f1_default'],
        'F1_Optimized': results['f1_optimized'],
        'F1_Improvement': results['f1_optimized'] - results['f1_default'],
        'Precision_Optimized': results['precision_optimized'],
        'Recall_Optimized': results['recall_optimized'],
        'AUC': results['auc'],
        'CV_F1_Mean': results['cv_f1_mean'],
        'CV_F1_Std': results['cv_f1_std']
    }
    for condition, results in optimized_results.items()
])

optimized_df.to_csv('/Users/eb2007/playground/bullpy/c4_experiments/data/processed/autism_optimized_results.csv', index=False)
print(f"\nOptimized results saved to: /Users/eb2007/playground/bullpy/c4_experiments/data/processed/autism_optimized_results.csv")

print("\n=== OPTIMIZATION COMPLETE ===")

# more attempts to improve models 

# smote 

In [ ]:
# SMOTE FOR ALL CONDITIONS
print("=== SMOTE CLASS IMBALANCE OPTIMIZATION (ALL CONDITIONS) ===")

from imblearn.over_sampling import SMOTE
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score

# Test ALL conditions
all_conditions = ['ADHD_target', 'Bipolar_target', 'Depression_target', 'Learning_target', 'OCD_target', 'Schizophrenia_target', 'Any_condition_target']
smote_results = {}

for condition in all_conditions:
    print(f"\nSMOTE OPTIMIZATION FOR {condition}")
    print("="*50)
    
    # Use the existing data preparation function from your notebook
    data_result = prepare_data_for_condition_clean(df_autism_only, condition, min_samples=500)
    
    if data_result[0] is not None:
        X_train, X_test, y_train, y_test, feature_names = data_result
        
        print(f"Original class distribution:")
        print(f"  Class 0: {sum(y_train == 0)} ({sum(y_train == 0)/len(y_train)*100:.1f}%)")
        print(f"  Class 1: {sum(y_train == 1)} ({sum(y_train == 1)/len(y_train)*100:.1f}%)")
        
        # 1. SMOTE
        print("\n1. Applying SMOTE...")
        try:
            # Use smaller k_neighbors for rare conditions
            k_neighbors = min(3, sum(y_train == 1)-1)
            smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
            X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
            
            print(f"After SMOTE class distribution:")
            print(f"  Class 0: {sum(y_train_smote == 0)} ({sum(y_train_smote == 0)/len(y_train_smote)*100:.1f}%)")
            print(f"  Class 1: {sum(y_train_smote == 1)} ({sum(y_train_smote == 1)/len(y_train_smote)*100:.1f}%)")
            
            # 2. Train model with SMOTE
            if condition == 'Schizophrenia_target':
                model = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1, class_weight='balanced')
            else:
                model = lgb.LGBMClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1, class_weight='balanced')
            
            model.fit(X_train_smote, y_train_smote)
            
            # 3. Evaluate
            y_probs = model.predict_proba(X_test)[:, 1]
            y_pred = model.predict(X_test)
            
            f1_smote = f1_score(y_test, y_pred)
            auc_smote = roc_auc_score(y_test, y_probs)
            precision_smote = precision_score(y_test, y_pred)
            recall_smote = recall_score(y_test, y_pred)
            
            # 4. Compare with baseline (no SMOTE)
            model_baseline = model.__class__(**model.get_params())
            model_baseline.fit(X_train, y_train)
            y_pred_baseline = model_baseline.predict(X_test)
            f1_baseline = f1_score(y_test, y_pred_baseline)
            auc_baseline = roc_auc_score(y_test, model_baseline.predict_proba(X_test)[:, 1])
            
            print(f"\nPERFORMANCE COMPARISON:")
            print(f"Baseline (no SMOTE): F1={f1_baseline:.4f}, AUC={auc_baseline:.4f}")
            print(f"SMOTE: F1={f1_smote:.4f}, AUC={auc_smote:.4f}")
            print(f"Improvement: F1 +{f1_smote-f1_baseline:.4f}, AUC +{auc_smote-auc_baseline:.4f}")
            
            smote_results[condition] = {
                'baseline_f1': f1_baseline,
                'smote_f1': f1_smote,
                'improvement': f1_smote - f1_baseline,
                'baseline_auc': auc_baseline,
                'smote_auc': auc_smote,
                'precision': precision_smote,
                'recall': recall_smote
            }
            
        except Exception as e:
            print(f"SMOTE failed for {condition}: {e}")
            print("Skipping this condition...")
            smote_results[condition] = {'error': str(e)}
        
    else:
        print(f"Skipping {condition} - insufficient samples")

# Summary
print("\n=== SMOTE RESULTS SUMMARY (ALL CONDITIONS) ===")
for condition, results in smote_results.items():
    if 'error' not in results:
        print(f"{condition}: F1 {results['baseline_f1']:.4f} → {results['smote_f1']:.4f} (+{results['improvement']:.4f})")
    else:
        print(f"{condition}: Failed - {results['error']}")

print("\n=== SMOTE OPTIMIZATION COMPLETE ===")

# advanced ensemble

In [ ]:
# ADVANCED ENSEMBLE METHODS
print("=== ADVANCED ENSEMBLE OPTIMIZATION ===")

from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Focus on best performing conditions
ensemble_conditions = ['Any_condition_target', 'Depression_target', 'Schizophrenia_target']
ensemble_results = {}

for condition in ensemble_conditions:
    print(f"\nADVANCED ENSEMBLE FOR {condition}")
    print("="*50)
    
    # Use the existing data preparation function from your notebook
    data_result = prepare_data_for_condition_clean(df_autism_only, condition, min_samples=500)
    
    if data_result[0] is not None:
        X_train, X_test, y_train, y_test, feature_names = data_result
        
        # 1. Stacking Classifier
        print("1. Training Stacking Classifier...")
        
        base_models = [
            ('rf', RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced')),
            ('lgb', lgb.LGBMClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1, class_weight='balanced'))
        ]
        
        meta_model = LogisticRegression(random_state=42, max_iter=1000)
        
        stacking = StackingClassifier(
            estimators=base_models,
            final_estimator=meta_model,
            cv=3,
            n_jobs=-1
        )
        
        stacking.fit(X_train, y_train)
        
        # 2. Weighted Voting Classifier
        print("2. Training Weighted Voting Classifier...")
        
        # Optimize weights (simplified)
        best_weights = None
        best_f1 = 0
        
        weight_combinations = [
            [0.5, 0.5],
            [0.6, 0.4],
            [0.4, 0.6],
            [0.7, 0.3],
            [0.3, 0.7]
        ]
        
        for weights in weight_combinations:
            voting = VotingClassifier(
                estimators=base_models,
                voting='soft',
                weights=weights
            )
            voting.fit(X_train, y_train)
            y_pred_vote = voting.predict(X_test)
            f1_vote = f1_score(y_test, y_pred_vote)
            
            if f1_vote > best_f1:
                best_f1 = f1_vote
                best_weights = weights
        
        # Use best weights
        voting_best = VotingClassifier(
            estimators=base_models,
            voting='soft',
            weights=best_weights
        )
        voting_best.fit(X_train, y_train)
        
        # 3. Evaluate both ensemble methods
        print("3. Evaluating ensemble methods...")
        
        # Stacking
        y_pred_stack = stacking.predict(X_test)
        y_probs_stack = stacking.predict_proba(X_test)[:, 1]
        f1_stack = f1_score(y_test, y_pred_stack)
        auc_stack = roc_auc_score(y_test, y_probs_stack)
        
        # Voting
        y_pred_vote = voting_best.predict(X_test)
        y_probs_vote = voting_best.predict_proba(X_test)[:, 1]
        f1_vote = f1_score(y_test, y_pred_vote)
        auc_vote = roc_auc_score(y_test, y_probs_vote)
        
        # 4. Compare with best single model
        if condition == 'Schizophrenia_target':
            single_model = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1, class_weight='balanced')
        else:
            single_model = lgb.LGBMClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1, class_weight='balanced')
        
        single_model.fit(X_train, y_train)
        y_pred_single = single_model.predict(X_test)
        f1_single = f1_score(y_test, y_pred_single)
        auc_single = roc_auc_score(y_test, single_model.predict_proba(X_test)[:, 1])
        
        print(f"\nPERFORMANCE COMPARISON:")
        print(f"Single model: F1={f1_single:.4f}, AUC={auc_single:.4f}")
        print(f"Stacking: F1={f1_stack:.4f}, AUC={auc_stack:.4f}")
        print(f"Voting (weights {best_weights}): F1={f1_vote:.4f}, AUC={auc_vote:.4f}")
        print(f"Best ensemble improvement: F1 +{max(f1_stack, f1_vote)-f1_single:.4f}")
        
        ensemble_results[condition] = {
            'single_f1': f1_single,
            'stacking_f1': f1_stack,
            'voting_f1': f1_vote,
            'best_ensemble_f1': max(f1_stack, f1_vote),
            'improvement': max(f1_stack, f1_vote) - f1_single,
            'best_weights': best_weights
        }
        
    else:
        print(f"Skipping {condition} - insufficient samples")

# Summary
print("\n=== ADVANCED ENSEMBLE RESULTS SUMMARY ===")
for condition, results in ensemble_results.items():
    print(f"{condition}: F1 {results['single_f1']:.4f} → {results['best_ensemble_f1']:.4f} (+{results['improvement']:.4f})")
    print(f"  Best weights: {results['best_weights']}")

print("\n=== ADVANCED ENSEMBLE OPTIMIZATION COMPLETE ===")

# NN

In [ ]:
# NEURAL NETWORK USING SCIKIT-LEARN (NO TENSORFLOW)
print("=== NEURAL NETWORK OPTIMIZATION (SKLEARN) ===")

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# Focus on best performing conditions
nn_conditions = ['Any_condition_target', 'Depression_target', 'Schizophrenia_target']
nn_results = {}

for condition in nn_conditions:
    print(f"\nNEURAL NETWORK FOR {condition}")
    print("="*50)
    
    # Prepare data using existing function
    data_result = prepare_data_for_condition_clean(df_autism_only, condition, min_samples=500)
    
    if data_result[0] is not None:
        X_train, X_test, y_train, y_test, feature_names = data_result
        
        # Scale features for neural network
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # 1. Build neural network
        print("1. Building neural network...")
        
        nn_model = MLPClassifier(
            hidden_layer_sizes=(128, 64, 32),
            activation='relu',
            solver='adam',
            alpha=0.001,
            learning_rate='adaptive',
            max_iter=500,
            random_state=42,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=10
        )
        
        # 2. Train neural network
        print("2. Training neural network...")
        nn_model.fit(X_train_scaled, y_train)
        
        # 3. Evaluate
        print("3. Evaluating neural network...")
        
        y_probs_nn = nn_model.predict_proba(X_test_scaled)[:, 1]
        y_pred_nn = nn_model.predict(X_test_scaled)
        
        f1_nn = f1_score(y_test, y_pred_nn)
        auc_nn = roc_auc_score(y_test, y_probs_nn)
        precision_nn = precision_score(y_test, y_pred_nn)
        recall_nn = recall_score(y_test, y_pred_nn)
        
        # 4. Compare with best tree-based model
        if condition == 'Schizophrenia_target':
            tree_model = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1, class_weight='balanced')
        else:
            tree_model = lgb.LGBMClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1, class_weight='balanced')
        
        tree_model.fit(X_train, y_train)
        y_pred_tree = tree_model.predict(X_test)
        f1_tree = f1_score(y_test, y_pred_tree)
        auc_tree = roc_auc_score(y_test, tree_model.predict_proba(X_test)[:, 1])
        
        print(f"\nPERFORMANCE COMPARISON:")
        print(f"Tree-based model: F1={f1_tree:.4f}, AUC={auc_tree:.4f}")
        print(f"Neural network: F1={f1_nn:.4f}, AUC={auc_nn:.4f}")
        print(f"Improvement: F1 +{f1_nn-f1_tree:.4f}, AUC +{auc_nn-auc_tree:.4f}")
        
        nn_results[condition] = {
            'tree_f1': f1_tree,
            'nn_f1': f1_nn,
            'improvement': f1_nn - f1_tree,
            'tree_auc': auc_tree,
            'nn_auc': auc_nn,
            'precision': precision_nn,
            'recall': recall_nn
        }
        
    else:
        print(f"Skipping {condition} - insufficient samples")

# Summary
print("\n=== NEURAL NETWORK RESULTS SUMMARY ===")
for condition, results in nn_results.items():
    print(f"{condition}: F1 {results['tree_f1']:.4f} → {results['nn_f1']:.4f} (+{results['improvement']:.4f})")

print("\n=== NEURAL NETWORK OPTIMIZATION COMPLETE ===")